## MapQC on Santamaria mapping

In [1]:
import scanpy as sc
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors

sys.path.append("../../")
sys.path.append("../utils/HECA")

import mapqc
import data_cleanup

In [2]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

In [3]:
batch_covariate = "sample"
kl = "1"
distance_type = "edistance"

In [4]:
path_emb = f"/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HECA/HECA_Santamaria_batch_{batch_covariate}_kl{kl}_latent.h5ad"
path_counts = (
    "/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HECA/HECA_Santamaria.h5ad"
)
path_results_dir = f"/lustre/groups/ml01/code/lisa.sikkema/mapqc/results/HECA/batch_{batch_covariate}_kl{kl}/mapqc"
path_figure_dir = "/lustre/groups/ml01/code/lisa.sikkema/mapqc_figures/heca_mapping/"

In [5]:
path_results_dir

'/lustre/groups/ml01/code/lisa.sikkema/mapqc/results/HECA/batch_sample_kl1/mapqc'

In [6]:
path_emb

'/lustre/groups/ml01/workspace/lisa.sikkema/mapqc/data/HECA/HECA_Santamaria_batch_sample_kl1_latent.h5ad'

check if results dir exists, if it's parent directory exists then create the final ("mapqc") directory

In [7]:
# check if parent directory exists:
parent_dir = os.path.dirname(path_results_dir)
if not os.path.isdir(parent_dir):
    raise ValueError(
        f"Parent directory of the specified results directory ({parent_dir}) does not exist."
    )
else:
    if not os.path.isdir(path_results_dir):
        # create directory:
        print(f"Creating directory {path_results_dir}...")
        os.makedirs(path_results_dir)

In [8]:
ref_q_key = "r_or_q"
ref_cat = "r"
query_cat = "q"

In [9]:
adata_emb = sc.read_h5ad(path_emb)
adata_counts = sc.read_h5ad(path_counts)

In [10]:
adata = data_cleanup.add_mapping_info_to_counts_adata(
    adata_counts, adata_emb, celltype_col="celltype"
)

add some ct info and order/clean up:

In [11]:
for col in ["celltype", "celltype_merged", "celltype_query"]:
    adata.obs[col] = [
        f"{label[0].upper()}{label[1:]}" if isinstance(label, str) else label
        for label in adata.obs[col]
    ]

In [12]:
ct_to_lineage_df = adata.obs.groupby("celltype").agg({"lineage": "first"})
ct_to_lineage_df["celltype"] = ct_to_lineage_df.index
ct_to_lineage_df.index.name = None
ct_to_lineage_df.sort_values(by=["lineage", "celltype"], inplace=True)
ct_to_lineage_mapper = {
    ct: lin for ct, lin in zip(ct_to_lineage_df.index, ct_to_lineage_df.lineage)
}
adata.obs["lineage_merged"] = adata.obs.celltype_merged.map(ct_to_lineage_mapper)

In [13]:
adata.obs["celltype_merged"] = pd.Categorical(
    adata.obs.celltype_merged,
    categories=ct_to_lineage_df.celltype.tolist(),
    ordered=False,
)

In [14]:
adata.obs["donor_query"] = [
    None
    if r_or_q == "r"
    else (
        s.split("-")[-3]
        if (group == "WOI Control") or ("biop" in s)
        else s.split("-")[-2]
    )
    for r_or_q, s, group in zip(adata.obs.r_or_q, adata.obs["sample"], adata.obs.Group)
]

In [15]:
del adata_counts

In [16]:
k = 500
max_k = 5000

In [29]:
# results_healthy = mapqc.MapqcResults(
#     adata=adata,
#     ref_q_key="r_or_q",
#     ref_cat="r",
#     query_cat="q",
#     condition_key="Group",
#     control_cats=["Ctrl_Superficial", "Ctrl_whole_uterus", "WOI Control"],
#     control_or_case="control",
#     case_cats=["AS"],
#     emb_key="X_scanvi_emb",
#     dataset_key="dataset",
#     sample_key="sample",
#     sampling_group_cat="leiden",
#     seed=0,
#     distance_type=distance_type,
#     n_cores=2,
#     k=k,
#     max_k=max_k
#     # n_nhoods=4,
# )
# results_healthy.run_mapqc()
# results_disease = mapqc.MapqcResults(
#     adata=adata,
#     ref_q_key="r_or_q",
#     ref_cat="r",
#     query_cat="q",
#     condition_key="Group",
#     control_cats=["Ctrl_Superficial", "Ctrl_whole_uterus", "WOI Control"],
#     control_or_case="case",
#     case_cats=["AS"],
#     emb_key="X_scanvi_emb",
#     dataset_key="dataset",
#     sample_key="sample",
#     sampling_group_cat="leiden",
#     seed=0,
#     distance_type=distance_type,
#     n_cores=2,
#     k=k,
#     max_k=max_k
#     # n_nhoods=4,
# )

# results_disease.run_mapqc()

store results:

In [30]:
# results_healthy.save(os.path.join(path_results_dir, f"control_{distance_type}_k{k}.pkl"))
# results_disease.save(os.path.join(path_results_dir, f"case_{distance_type}_k{k}.pkl"))

load results if starting from here:

In [17]:
results_healthy = mapqc.load(
    os.path.join(path_results_dir, f"control_{distance_type}_k{k}.pkl")
)
results_disease = mapqc.load(
    os.path.join(path_results_dir, f"case_{distance_type}_k{k}.pkl")
)

In [20]:
results_healthy.result_stats

{'n_nhoods': 749,
 'k': 500,
 'max_k': 5000,
 'perc_of_nhoods_passed_filter': 85,
 'n_samples_total': 6,
 'n_samples_with_results': 6,
 'n_samples_not_mixing_with_ref': 0,
 'perc_samples_not_mixing_with_ref': 0,
 'perc_cells_sampled': 96.6,
 'perc_cells_with_results': 94.89,
 'perc_non_mixing_cells': 6.07}

In [21]:
results_disease.result_stats

{'n_nhoods': 835,
 'k': 500,
 'max_k': 5000,
 'perc_of_nhoods_passed_filter': 97,
 'n_samples_total': 17,
 'n_samples_with_results': 17,
 'n_samples_not_mixing_with_ref': 2,
 'perc_samples_not_mixing_with_ref': 12,
 'perc_cells_sampled': 95.26,
 'perc_cells_with_results': 92.5,
 'perc_non_mixing_cells': 11.39}